In [1]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

# Siyah-beyaz resimleri RGB'ye çeviren ve boyutlandıran dönüşümler
train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Caltech-101 veri setini indir
full_dataset = datasets.Caltech101(root="./data", download=True)
class_names = full_dataset.categories

# %70 Train, %15 Validation, %15 Test bölme
total_len = len(full_dataset)
train_len = int(0.70 * total_len)
val_len = int(0.15 * total_len)
test_len = total_len - train_len - val_len

generator = torch.Generator().manual_seed(42)
train_raw, val_raw, test_raw = random_split(full_dataset, [train_len, val_len, test_len], generator=generator)

# Her alt kümeye kendi transformunu bağımsız atamak için sarmalayıcı (Wrapper)
class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

train_dataset = SubsetWithTransform(train_raw, train_transform)
val_dataset = SubsetWithTransform(val_raw, eval_transform)
test_dataset = SubsetWithTransform(test_raw, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Sınıf Sayısı: {len(class_names)} | Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Kullanılan cihaz: cpu


100%|██████████| 137M/137M [00:07<00:00, 18.0MB/s]


Sınıf Sayısı: 101 | Train: 6073, Val: 1301, Test: 1303


In [2]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Önceki katmanları dondur (Hızlı ve kararlı eğitim için)
for param in model.parameters():
    param.requires_grad = False

# Son katmanı 101 sınıfa göre uyarla
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 6

best_val_acc = 0.0
start_train_time = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_resnet18_caltech101.pth")

total_train_time = time.time() - start_train_time
print(f"\nToplam Eğitim Süresi: {total_train_time:.2f} saniye | En İyi Val Accuracy: {best_val_acc:.4f}")

KeyboardInterrupt: 

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchvision import models
from sklearn.metrics import accuracy_score
import time

# 1. Pretrained ResNet-18 yükle ve son katmanı çıkar (Özellik çıkarıcı olarak kullan)
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = backbone.fc.in_features
backbone.fc = nn.Identity()
backbone = backbone.to(device)
backbone.eval()

start_train_time = time.time()

# 2. Görüntü özelliklerini tek seferde çıkar (CPU'da ~40 sn sürer)
def extract_features(loader):
    feats, lbls = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            out = backbone(images)
            feats.append(out.cpu())
            lbls.append(labels)
    return torch.cat(feats), torch.cat(lbls)

print("Görüntü özellikleri çıkarılıyor (yaklaşık 40 saniye)...")
X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)
X_test, y_test = extract_features(test_loader)

# 3. Sadece son sınıflandırıcı katmanı eğit (2-3 saniye sürer)
classifier = nn.Linear(num_ftrs, len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 15

feat_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

best_val_acc = 0.0
for epoch in range(epochs):
    classifier.train()
    running_loss = 0.0
    for bx, by in feat_train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        outputs = classifier(bx)
        loss = criterion(outputs, by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    classifier.eval()
    with torch.no_grad():
        val_outputs = classifier(X_val.to(device))
        val_preds = torch.argmax(val_outputs, dim=1).cpu().numpy()
        val_acc = accuracy_score(y_val.numpy(), val_preds)

    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(feat_train_loader):.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(classifier.state_dict(), "best_classifier_caltech101.pth")

# Tam modeli (backbone + en iyi classifier) birleştir
classifier.load_state_dict(torch.load("best_classifier_caltech101.pth"))
backbone.fc = classifier
model = backbone
torch.save(model.state_dict(), "best_resnet18_caltech101.pth")

total_train_time = time.time() - start_train_time
print(f"\nToplam Eğitim Süresi: {total_train_time:.2f} saniye | En İyi Val Accuracy: {best_val_acc:.4f}")

Görüntü özellikleri çıkarılıyor (yaklaşık 40 saniye)...


KeyboardInterrupt: 

In [4]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

# 1. Veri Seti (CIFAR-10 - RAM üzerinden anında okunur)
transform = transforms.Compose([
    transforms.Resize((128, 128)), # CPU'da süper hızlı olması için 128x128
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
class_names = full_train.classes

# Hızlı eğitim için dengeli alt küme (Train: 8000, Val: 2000, Test: 2000)
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, _ = random_split(full_train, [8000, 2000, 40000], generator=generator)
test_subset, _ = random_split(test_dataset, [2000, 8000], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

# 2. Model (ResNet-18 Özellik Çıkarımı + Sınıflandırıcı)
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = backbone.fc.in_features
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()

start_train_time = time.time()

def extract_features(loader):
    feats, lbls = [], []
    with torch.no_grad():
        for images, labels in loader:
            out = backbone(images.to(device))
            feats.append(out.cpu())
            lbls.append(labels)
    return torch.cat(feats), torch.cat(lbls)

print("Özellikler çıkarılıyor (yaklaşık 15-20 saniye)...")
X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)
X_test, y_test = extract_features(test_loader)

classifier = nn.Linear(num_ftrs, len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 15

feat_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
best_val_acc = 0.0

for epoch in range(epochs):
    classifier.train()
    running_loss = 0.0
    for bx, by in feat_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(classifier(bx), by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    classifier.eval()
    with torch.no_grad():
        val_preds = torch.argmax(classifier(X_val.to(device)), dim=1).cpu().numpy()
        val_acc = accuracy_score(y_val.numpy(), val_preds)

    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(feat_loader):.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(classifier.state_dict(), "best_cls.pth")

classifier.load_state_dict(torch.load("best_cls.pth"))
backbone.fc = classifier
model = backbone.eval()
torch.save(model.state_dict(), "best_resnet18_cifar10.pth")
total_train_time = time.time() - start_train_time

# 3. Test Metrikleri ve Süre Ölçümü
test_preds, test_targets, inf_times = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        t0 = time.time()
        outputs = model(images.to(device))
        t1 = time.time()
        inf_times.append((t1 - t0) / images.size(0) * 1000)
        test_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        test_targets.extend(labels.numpy())

acc = accuracy_score(test_targets, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_targets, test_preds, average="macro")
print(f"\n--- SONUÇLAR ---")
print(f"Toplam Eğitim Süresi : {total_train_time:.2f} saniye")
print(f"Test Accuracy        : {acc:.4f}")
print(f"Macro Precision      : {prec:.4f}")
print(f"Macro Recall         : {rec:.4f}")
print(f"Macro F1-Score       : {f1:.4f}")
print(f"Tahmin Süresi (Inf.) : {np.mean(inf_times):.2f} ms / görüntü")

Kullanılan cihaz: cpu


100%|██████████| 170M/170M [17:04<00:00, 166kB/s]


Özellikler çıkarılıyor (yaklaşık 15-20 saniye)...
Epoch [1/15] - Loss: 0.8781 - Val Acc: 0.8095
Epoch [3/15] - Loss: 0.4375 - Val Acc: 0.8310
Epoch [6/15] - Loss: 0.3600 - Val Acc: 0.8355
Epoch [9/15] - Loss: 0.3180 - Val Acc: 0.8320
Epoch [12/15] - Loss: 0.2903 - Val Acc: 0.8345
Epoch [15/15] - Loss: 0.2721 - Val Acc: 0.8340

--- SONUÇLAR ---
Toplam Eğitim Süresi : 400.79 saniye
Test Accuracy        : 0.8445
Macro Precision      : 0.8452
Macro Recall         : 0.8449
Macro F1-Score       : 0.8440
Tahmin Süresi (Inf.) : 31.13 ms / görüntü


In [5]:
!pip install -q gradio
import gradio as gr
from PIL import Image

def predict_cifar(image):
    if image is None:
        return None
    img = Image.fromarray(image).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(img_tensor), dim=1)[0]
    top3_prob, top3_idx = torch.topk(probs, 3)
    return {class_names[idx]: float(prob) for prob, idx in zip(top3_prob, top3_idx)}

demo = gr.Interface(
    fn=predict_cifar,
    inputs=gr.Image(label="Fotoğraf Yükleyin"),
    outputs=gr.Label(num_top_classes=3, label="Model Tahmini (Top-3)"),
    title="CIFAR-10 Görüntü Sınıflandırıcı (ResNet-18)",
    description="Sınıflar: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4cef28b1474098bdb5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
